In [1]:
#!pip install -qq numpy pandas

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

### Load DataFrames

In [3]:
# This assumes there is a /data/ folder in the current working directory
#  containing subfolders with .csv files. Every .csv file in any
#  subdirectory is loaded to this dictionary as a DataFrame. All dtypes
#  are initially strings.

DFDICT = {
    subdir.name: {
        csv.stem: pd.read_csv(csv, dtype="string", low_memory=False)
        for csv in subdir.glob("*.csv")
    }
    for subdir in (Path.cwd() / "data").iterdir() if subdir.is_dir()
}

### Infer Data Types

In [4]:
# Regex expressions to determine if a string looks like 
#  an integer or float type.
INT_RE = r"[+-]?(?:0|[1-9]\d*|[1-9]\d{0,2}(?:,\d{3})+)"
FLOAT_RE = r"[+-]?(?:0\.\d*|[1-9]\d*(?:\.\d*)?|\.\d+)"

DTYPE_RULE_MAP = {
    "EMPTY":    lambda lst: lst[-1] == "NaN",
    "DATETIME": lambda lst: lst[-1] == "DateTime",
    "STRING":   lambda lst: lst[-1] in ["string", "Heading", "Categorical"],
    "INTEGER":  lambda lst: "Int64" in lst,
    "FLOAT":    lambda lst: "Float64" in lst,
}

def inferDataTypes(df: pd.DataFrame) -> dict[str, list[str]]:
    """
    This function returns a dictionary with a value for easch column name
    in the DataFrame argument and a list of inferred information about
    the type of data in the column.
    """

    # This dictionary will be returned.
    dtypeMap = { col: [] for col in df.columns }
    
    for col in df.columns:
        
        # Strip leading/trailing whitespace and drop all duplicate
        #  and missing values (missing either N/A or empty string).        
        coldata = df[col].str.strip().drop_duplicates().dropna()
        coldata = coldata[coldata != ""]

        # First checks. 
        # Label a column as a string dtype if:
        #  - it is empty
        #  - it holds datetime data
        #  - it holds wind direction heading values
        #  - if it has been specifically chosen as categorical
        categoryCols = ["STATION"]
        if coldata.empty:
            dtypeMap[col] = ["string", "NaN"]
        elif (
          col.startswith("Sun") or 
          col.rstrip("0123456789").lower().endswith(("date", "time", "zone"))
        ):
            dtypeMap[col] = ["string", "DateTime"]
        elif col.endswith("Direction"):
            dtypeMap[col] = ["string", "Heading"]
        elif col in categoryCols:
            dtypeMap[col] = ["string", "Categorical"]

        # Second pass.
        #  - Skip if the first pass labelled it a string already.
        #  - Otherwise check if unique values look numeric.
        #  - If not, label it a string, but then strip the last
        #     character from each unique value and check again
        #     if it looks numeric. If so, append the appropriate 
        #     numeric dtype to the list of type information in the 
        #     dictionary.
        #
        # The last check here is to catch numeric columns that have
        #  occasional one-letter entries or suffixes, as is standard
        #  in weather data formatting.
        #  - For example, precipitation might just say "T" for trace or have
        #     an "s" at the end to indicate snow. Visibility may have a "V"
        #     as a suffix to indicate variability.
        if dtypeMap[col]:
            pass
        elif coldata.str.fullmatch(INT_RE).all():
            dtypeMap[col] = ["Int64"]
        elif coldata.str.fullmatch(FLOAT_RE).all(): 
            dtypeMap[col] = ["Float64"] 
        else:
            dtypeMap[col] = ["string"]

        if (
          dtypeMap[col] == ["string"] 
          and coldata.str[-1].drop_duplicates().str.isalnum().all()
        ):
            coldata = coldata.str[:-1].drop_duplicates()
            coldata = coldata[coldata != ""]
            if coldata.empty:
                pass
            elif coldata.str.fullmatch(INT_RE).all(): 
                dtypeMap[col].append("Int64")
            elif coldata.str.fullmatch(FLOAT_RE).all(): 
                dtypeMap[col].append("Float64")

    return dtypeMap

### Datatype Inference on Weather Datasets

In [5]:
# Infer data types for every column in every dataset
#  of the dataframe dictionary.

df_dtypeDict = {
    _dir : { n : inferDataTypes(d) for n, d in _dict.items() }
    for _dir, _dict in DFDICT.items()
}

In [6]:
# This confirms that the column sets are identical across weather datasets,
#  and that inferred dtypes are the same for each column for each dataset.

nWeatherDFs = len(df_dtypeDict["weather"])
wColCounts = {}
for d in df_dtypeDict["weather"].values():
    for c in d:
        wColCounts[c] = wColCounts.get(c, 0) + 1
print(
  'No. of columns shared between all weather datasets:',
  len([ k for k, v in wColCounts.items() if v == nWeatherDFs ])
)

wDiffTypeCols = {}
for c in wColCounts.keys():
    colInfo = { n : d for n, d[c] in df_dtypeDict["weather"].items() }
    if len({tuple(typeList) for typeList in colInfo.values()}) != 1:
        wDiffTypeCols[c] = colInfo
print(
  'No. of columns with discrepancies in inferred dtype info between datasets:',
  len(wDiffTypeCols)
)

No. of columns shared between all weather datasets: 125
No. of columns with discrepancies in inferred dtype info between datasets: 0


In [7]:
# Categorizing columns from the 2024 Weather Dataset based on inferred dtype info.

W24 = DFDICT['weather']['MIA_2024']
W24_dtypeDict = df_dtypeDict['weather']['MIA_2024']
print(W24.shape)

for _type, _rule in DTYPE_RULE_MAP.items():
    cols = [c for c in W24.columns if _rule(W24_dtypeDict[c])]
    print("\n" + "-"*30 + f"\n{_type} Columns: {len(cols)}\n" + "-"*30)
    if cols: print(*cols, sep="\n")

(13105, 125)

------------------------------
EMPTY Columns: 17
------------------------------
MonthlyAverageRH
MonthlyDewpointTemperature
MonthlyGreatestSnowDepth
MonthlyGreatestSnowDepthDate
MonthlyGreatestSnowfall
MonthlyGreatestSnowfallDate
MonthlyTotalSnowfall
MonthlyWetBulb
BackupDirection
BackupDistance
BackupDistanceUnit
BackupElements
BackupElevation
BackupEquipment
BackupLatitude
BackupLongitude
BackupName

------------------------------
DATETIME Columns: 21
------------------------------
DATE
Sunrise
Sunset
MonthlyGreatestPrecipDate
MonthlyMaxSeaLevelPressureValueDate
MonthlyMaxSeaLevelPressureValueTime
MonthlyMinSeaLevelPressureValueDate
MonthlyMinSeaLevelPressureValueTime
ShortDurationEndDate005
ShortDurationEndDate010
ShortDurationEndDate015
ShortDurationEndDate020
ShortDurationEndDate030
ShortDurationEndDate045
ShortDurationEndDate060
ShortDurationEndDate080
ShortDurationEndDate100
ShortDurationEndDate120
ShortDurationEndDate150
ShortDurationEndDate180
WindEquipmentChange

In [8]:
# Display unique values for each column from W24.

W24_uniqueVals = []
for col in W24.columns:
    coldata = W24[col].str.strip().drop_duplicates().dropna()
    coldata = coldata[coldata != ""]
    col_nunique = coldata.nunique()
    if col_nunique > 0:
        unique_vals = coldata.unique()
        W24_uniqueVals.append((col_nunique, W24_dtypeDict[col][0], col, unique_vals))
W24_uniqueVals.sort(key=lambda x: x[0])

# Display a specific column or columns of a specified inferred dtype and 
#  with a cap on how many unique values there are.

COL_DTYPE = 'string'
COLNAME = ''
for n, dtype, col, unique_vals in W24_uniqueVals:
    if col == COLNAME or dtype == COL_DTYPE:
        print(col, W24_dtypeDict[col])
        print(unique_vals, '\n')

STATION ['string', 'Categorical']
<StringArray>
['72202012839']
Length: 1, dtype: string 

NAME ['string']
<StringArray>
['MIAMI INTERNATIONAL AIRPORT, FL US']
Length: 1, dtype: string 

WindEquipmentChangeDate ['string', 'DateTime']
<StringArray>
['2009-07-14']
Length: 1, dtype: string 

SOURCE ['string']
<StringArray>
['7', '4', '6', 'O']
Length: 4, dtype: string 

REPORT_TYPE ['string']
<StringArray>
['FM-15', 'FM-12', 'SOD', 'FM-16', 'SOM', 'SY-MT']
Length: 6, dtype: string 

MonthlyMaxSeaLevelPressureValueTime ['string', 'DateTime']
<StringArray>
['1109', '1127', '1053', '0953', '2253', '2344', '1041', '1004']
Length: 8, dtype: string 

MonthlyMinSeaLevelPressureValueDate ['string', 'DateTime']
<StringArray>
['09', '05', '23', '03', '20', '06', '07', '26', '21']
Length: 9, dtype: string 

MonthlyGreatestPrecipDate ['string', 'DateTime']
<StringArray>
['11-11', '18-19', '22-23', '30-30', '29-30', '11-12', '28-29', '24-24',
 '01-02', '04-05']
Length: 10, dtype: string 

MonthlyMaxSe

### Datatype Inference on Electricity Data

In [9]:
# Get electricity datasets determine which columns they have
#  in common and which 

FLA = DFDICT['electricity']['Region_FLA']
FPL = DFDICT['electricity']['FPL']
FLAtypeInfo = df_dtypeDict['electricity']['Region_FLA']
FPLtypeInfo = df_dtypeDict['electricity']['FPL']

FLA_uniqueCols = [c for c in FLA.columns if c not in FPL.columns]
FPL_uniqueCols = [c for c in FPL.columns if c not in FLA.columns]
E_sharedCols = [c for c in FLA.columns if c not in FLA_uniqueCols]

print(FLA.shape); print(FPL.shape)

(90504, 66)
(90504, 89)


In [10]:
# List, by datatype, columns shared by both datasets

E_sharedColTypes = { 
    c : { 'FLA' : next( t for t, r in DTYPE_RULE_MAP.items() if r(FLAtypeInfo[c]) ),
          'FPL' : next( t for t, r in DTYPE_RULE_MAP.items() if r(FPLtypeInfo[c]) )  }
    for c in E_sharedCols
}

print("="*15+ f" SHARED COLUMNS: {len(E_sharedCols)} " +"="*15+"\n"+"="*50)
for _type, _rule in DTYPE_RULE_MAP.items():
    
    cols, adds = [], []
    for c, d in E_sharedColTypes.items():
        if _type == d["FLA"] == d["FPL"]:
            cols.append(c)
            if _type == "EMPTY": adds.append("EMPTY in Both")
        elif _type == "EMPTY" and _type in d.values(): 
            cols.append(c)
            adds.append( 
              f"{d['FLA']} in FLA " if d['FLA'] != _type else 
              f"{d['FPL']} in FPL "
            )
    
    print(f"\n{_type} Columns: {len(cols)}\n" + "-"*50)
    if cols: 
        print( 
          *[ c.ljust(34) + (a if adds else '').rjust(16) 
             for c, a in zip(cols, adds if adds else ['']*len(cols)) ], 
          sep="\n"
        )

=============== SHARED COLUMNS: 40 ===============

EMPTY Columns: 10
--------------------------------------------------
NG: COL                            INTEGER in FLA 
NG: GEO                              EMPTY in Both
NG: WAT                            INTEGER in FLA 
NG: PS                               EMPTY in Both
NG: WND                              EMPTY in Both
NG: WNB                              EMPTY in Both
NG: OES                              EMPTY in Both
NG: UES                              EMPTY in Both
NG: UNK                              EMPTY in Both
CO2 Emissions: COL                 INTEGER in FLA 

DATETIME Columns: 4
--------------------------------------------------
UTC time                                          
Local date                                        
Local time                                        
Time zone                                         

STRING Columns: 0
--------------------------------------------------

INTEGER Columns: 21
--

In [11]:
# List columns unique to the FLA dataset.

print("="*9+ f" FLA COLUMNS ONLY: {len(FLA_uniqueCols)} " +"="*9+"\n"+"="*40)
for _type, _rule in DTYPE_RULE_MAP.items():
    cols = [c for c in FLA_uniqueCols if _rule(FLAtypeInfo[c])]
    print(f"\n{_type} Columns: {len(cols)}\n" + "-"*40)
    if cols: print(*cols, sep="\n")

========= FLA COLUMNS ONLY: 26 =========

EMPTY Columns: 14
----------------------------------------
CAL
CAR
CENT
FLA
MIDA
MIDW
NE
NW
NY
SW
TEN
TEX
CAN
MEX

DATETIME Columns: 0
----------------------------------------

STRING Columns: 1
----------------------------------------
Region

INTEGER Columns: 11
----------------------------------------
Sum (NG)
Sum (Trade)
Sum (Imports)
Sum (Exports)
SE
Balance NG D TI
Balance TI Trade
Balance NG
Positive Gen from COL
Positive Gen from NG
Positive Gen from OIL

FLOAT Columns: 0
----------------------------------------


In [12]:
# List columns unique to the FPL dataset.

print("="*9+ f" FPL COLUMNS ONLY: {len(FPL_uniqueCols)} " +"="*9+"\n"+"="*40)
for _type, _rule in DTYPE_RULE_MAP.items():
    cols = [c for c in FPL_uniqueCols if _rule(FPLtypeInfo[c])]
    print(f"\n{_type} Columns: {len(cols)}\n" + "-"*40)
    if cols: print(*cols, sep="\n")

========= FPL COLUMNS ONLY: 49 =========

EMPTY Columns: 20
----------------------------------------
Imputed COL Gen
Imputed GEO Gen
Imputed WAT Gen
Imputed PS Gen
Imputed SNB Gen
Imputed WND Gen
Imputed WNB Gen
Imputed BAT Gen
Imputed OES Gen
Imputed UES Gen
Imputed UNK Gen
Adjusted COL Gen
Adjusted GEO Gen
Adjusted WAT Gen
Adjusted PS Gen
Adjusted WND Gen
Adjusted WNB Gen
Adjusted OES Gen
Adjusted UES Gen
Adjusted UNK Gen

DATETIME Columns: 0
----------------------------------------

STRING Columns: 2
----------------------------------------
BA
Generation only?

INTEGER Columns: 27
----------------------------------------
Imputed demand
Imputed net generation
Imputed total interchange
Adjusted demand
Adjusted net generation
Adjusted total interchange
Imputed NG Gen
Imputed NUC Gen
Imputed OIL Gen
Imputed SUN Gen
Imputed OTH Gen
Adjusted NG Gen
Adjusted NUC Gen
Adjusted OIL Gen
Adjusted SUN Gen
Adjusted SNB Gen
Adjusted BAT Gen
Adjusted OTH Gen
FMPP
FPC
GVL
HST
JEA
NSB
SEC
SOCO
TEC

F